# EEG_40 — Caratterizzazione dei fenotipi a livello di **soggetto**, stratificata per **banda**

**Obiettivo (meeting Iacomi, giu 2026).** Finora i fenotipi C0/C1 sono stati *scoperti* e
*validati* a livello di trial (ARI trial-level, stabilità inter-sessione). Questo notebook
risponde alla domanda complementare: **cosa distingue neurofisiologicamente C0 da C1**, a
livello di soggetto e separando per banda di frequenza.

Pipeline (= paper Iacomi et al. 2026, di cui D. Uras è coautore):

1. per ogni soggetto, media **tutti** i trial `_img` → 1 profilo per soggetto;
2. media dentro il cluster → profilo C0 e profilo C1;
3. confronto C0 vs C1 con **effect size** (Cohen's d), non p-value (i p-value sono saturati
   su decine di migliaia di trial — cfr. EEG_23 §14).

Due famiglie di metriche, una per ciascuna ipotesi di fenotipo:

| Metrica | Cosa misura | Ipotesi |
|---------|-------------|---------|
| **band power** (Welch, relativa) | ampiezza spettrale per canale | C0 fronto-motor → ↑ banda motoria |
| **\|PCC\|** per banda | connettività di **ampiezza** | dovrebbe separare C0 |
| **PLV** per banda | connettività di **fase** | dovrebbe separare C1 (phase-coupling) |

> Bande (convenzione `scripts/features/bandpowers.py`): δ 1–4, θ 4–8, α 8–13, β 13–30,
> **γ 30–45 Hz** (tagliata a 45 per stare sotto il notch di rete a 50 Hz).

⚠️ **Gira sul server**: legge i CSV grezzi da `data/raw_csv/training_set/` (61×384, fs=256),
non presenti in questo worktree.

## §0 — Setup, path, montage

In [ ]:
import json, logging, math, re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import hilbert, butter, filtfilt, welch
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('eeg40')

project_root = next((p for p in [Path.cwd()] + list(Path.cwd().parents)
                     if (p / '.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures';            FIG_DIR.mkdir(exist_ok=True)
CKPT_DIR = project_root / 'models' / 'eeg40';   CKPT_DIR.mkdir(parents=True, exist_ok=True)
CSV_ROOT = project_root / 'data' / 'raw_csv' / 'training_set'
ELOC_FILE = project_root / 'src' / 'io' / 'ebneuro.locs'
assert CSV_ROOT.exists(), f'CSV_ROOT non trovato: {CSV_ROOT}'

FS       = 256        # Hz
N_CHAN   = 61
RANDOM_SEED = 42
BANDS = {'delta': (1, 4), 'theta': (4, 8), 'alpha': (8, 13),
         'beta': (13, 30), 'gamma': (30, 45)}
BAND_NAMES = list(BANDS)
log.info(f'project_root = {project_root}')
log.info(f'Bande: {BANDS}')

In [ ]:
# ── Montage autoritativo (identico a EEG_07f): allinea le 61 righe CSV alle
#    posizioni scalpo. ebneuro.locs ha 64 nomi → rename T3/T4/T5/T6 → drop A1/A2
#    → primi 61. Tutti i 61 canali risultanti hanno una posizione valida.
import mne
_RENAME = {'T3': 'T7', 'T4': 'T8', 'T5': 'P7', 'T6': 'P8'}
_BAD    = {'A1', 'A2'}
_mont   = mne.channels.read_custom_montage(str(ELOC_FILE), coord_frame='head')
_pos_all = _mont.get_positions()['ch_pos']
CHAN_NAMES = [_RENAME.get(ch, ch) for ch in _mont.ch_names if ch not in _BAD][:N_CHAN]
CH_POS     = {_RENAME.get(ch, ch): _pos_all[ch]
              for ch in _mont.ch_names if ch not in _BAD}
assert len(CHAN_NAMES) == N_CHAN, len(CHAN_NAMES)
assert all(ch in CH_POS for ch in CHAN_NAMES), 'canale senza posizione'

def make_topo_info(ch_names=None):
    """mne.Info con montage per plot_topomap."""
    ch = ch_names if ch_names is not None else CHAN_NAMES
    info = mne.create_info(list(ch), sfreq=FS, ch_types='eeg')
    dig  = mne.channels.make_dig_montage(ch_pos={c: CH_POS[c] for c in ch}, coord_frame='head')
    info.set_montage(dig, on_missing='warn')
    return info

log.info(f'Canali ({N_CHAN}): {CHAN_NAMES[:6]} ... {CHAN_NAMES[-3:]}')

## §1 — Label di cluster C0/C1 e indice dei trial `_img` per soggetto

I label vengono dall'export di EEG_16b (`configs/eeg16b_cluster_labels.json`):
C0 = Fronto-motor, C1 = Fronto-occipital. P022/P023 già esclusi a monte.

In [ ]:
# ── Cluster labels da EEG_16b ─────────────────────────────────────────────────
LBL_FILE = project_root / 'configs' / 'eeg16b_cluster_labels.json'
assert LBL_FILE.exists(), f'Label non trovati: {LBL_FILE} (esegui prima EEG_16b)'
_lbl = json.loads(LBL_FILE.read_text())
CLUSTER_OF = {int(s): int(l) for s, l in zip(_lbl['subj_ids'], _lbl['labels'])}
CLUSTER_NAMES = {0: 'Fronto-motor (C0)', 1: 'Fronto-occipital (C1)'}
n_c0 = sum(v == 0 for v in CLUSTER_OF.values())
n_c1 = sum(v == 1 for v in CLUSTER_OF.values())
log.info(f'Cluster: C0={n_c0} soggetti, C1={n_c1} soggetti (tot {len(CLUSTER_OF)})')

# ── Indice trial _img per soggetto (tutte le sessioni) ───────────────────────
_PAT = re.compile(r'^P(\d+)_S(\d+)$')
trials_of = defaultdict(list)
for sess_dir in sorted(CSV_ROOT.glob('P*_S*')):
    m = _PAT.match(sess_dir.name)
    if not m:
        continue
    sid = int(m.group(1))
    if sid not in CLUSTER_OF:        # soggetto senza label (es. escluso) → skip
        continue
    trials_of[sid].extend(sorted(sess_dir.glob('*_img.csv')))

SUBJECTS = sorted(trials_of)
_counts = {s: len(trials_of[s]) for s in SUBJECTS}
log.info(f'Soggetti con trial img: {len(SUBJECTS)}')
log.info(f'Trial/soggetto: min={min(_counts.values())} max={max(_counts.values())} '
         f'tot={sum(_counts.values())}')

## §2 — Helper: I/O, band-pass, band power, connettività

Le funzioni di connettività sono identiche a EEG_07f (stessa convenzione `hilbert(x, axis=1)`).
La band power è **relativa** (frazione di potenza nella banda): invariante allo z-score
per-canale, quindi confrontabile fra canali e soggetti.

In [ ]:
def load_trial(csv_path):
    """CSV (61, 384) float32, righe=canali."""
    return pd.read_csv(csv_path, header=None).values.astype(np.float32)

def normalize_trial(x):
    """Instance norm: z-score per-canale sull'asse temporale (Bomatter 2024)."""
    mean = x.mean(axis=1, keepdims=True)
    std  = x.std(axis=1, keepdims=True).clip(1e-6)
    return (x - mean) / std

# ── band-pass IIR zero-phase ─────────────────────────────────────────────────
_FILTERS = {name: butter(4, [lo / (FS / 2), hi / (FS / 2)], btype='band')
            for name, (lo, hi) in BANDS.items()}
def bandpass(x, band):
    b, a = _FILTERS[band]
    return filtfilt(b, a, x, axis=1).astype(np.float32)

# ── band power relativa per canale (Welch sul segnale full-band) ─────────────
def rel_bandpower(x):
    """Ritorna (n_band, N_CHAN): frazione di potenza per banda e canale."""
    freqs, psd = welch(x, fs=FS, nperseg=min(256, x.shape[1]), axis=1)  # (N, F)
    total = np.trapz(psd, freqs, axis=1) + 1e-12                        # (N,)
    out = np.empty((len(BANDS), x.shape[0]), dtype=np.float32)
    for k, (lo, hi) in enumerate(BANDS.values()):
        sel = (freqs >= lo) & (freqs <= hi)
        out[k] = np.trapz(psd[:, sel], freqs[sel], axis=1) / total
    return out

# ── connettività di ampiezza e fase ──────────────────────────────────────────
def abs_pcc(x):
    m = np.abs(np.corrcoef(x)).astype(np.float32)
    np.fill_diagonal(m, 0.0)
    return m

def plv(x):
    z = hilbert(x, axis=1)
    ph = z / (np.abs(z) + 1e-10)
    N = x.shape[0]
    out = np.zeros((N, N), dtype=np.float32)
    for i in range(N):
        out[i] = np.abs(np.mean(ph[i] * np.conj(ph), axis=1))
    np.fill_diagonal(out, 0.0)
    return out

print('Helper OK — bande:', BAND_NAMES)

## §3 — Feature per soggetto (media su tutti i trial img)

Per ogni soggetto e ogni banda accumuliamo: band power per canale, matrice |PCC| e matrice
PLV (calcolate sul segnale **filtrato in banda**). Tutto mediato sui trial del soggetto.
Risultato in cache su `models/eeg40/persubj_band_features.npz`.

In [ ]:
CACHE = CKPT_DIR / 'persubj_band_features.npz'
NB = len(BANDS)

if CACHE.exists():
    _z = np.load(CACHE, allow_pickle=True)
    POW  = _z['power']   # (S, NB, 61)
    PCC  = _z['pcc']     # (S, NB, 61, 61)
    PLV  = _z['plv']     # (S, NB, 61, 61)
    SUBJ = _z['subjects'].tolist()
    log.info(f'Cache caricata: POW{POW.shape}  PCC{PCC.shape}  PLV{PLV.shape}')
else:
    SUBJ = list(SUBJECTS)
    S = len(SUBJ)
    POW = np.zeros((S, NB, N_CHAN), dtype=np.float32)
    PCC = np.zeros((S, NB, N_CHAN, N_CHAN), dtype=np.float32)
    PLV = np.zeros((S, NB, N_CHAN, N_CHAN), dtype=np.float32)
    for si, sid in enumerate(tqdm(SUBJ, desc='soggetti')):
        paths = trials_of[sid]
        acc_pow = np.zeros((NB, N_CHAN), dtype=np.float64)
        acc_pcc = np.zeros((NB, N_CHAN, N_CHAN), dtype=np.float64)
        acc_plv = np.zeros((NB, N_CHAN, N_CHAN), dtype=np.float64)
        nt = 0
        for p in paths:
            try:
                x = normalize_trial(load_trial(p))
            except Exception as e:
                log.warning(f'skip {p.name}: {e}'); continue
            if x.shape[0] != N_CHAN:
                log.warning(f'skip {p.name}: shape {x.shape}'); continue
            acc_pow += rel_bandpower(x)
            for k, band in enumerate(BAND_NAMES):
                xb = bandpass(x, band)
                acc_pcc[k] += abs_pcc(xb)
                acc_plv[k] += plv(xb)
            nt += 1
        if nt == 0:
            log.warning(f'P{sid:03d}: 0 trial validi'); continue
        POW[si] = (acc_pow / nt).astype(np.float32)
        PCC[si] = (acc_pcc / nt).astype(np.float32)
        PLV[si] = (acc_plv / nt).astype(np.float32)
    np.savez_compressed(CACHE, power=POW, pcc=PCC, plv=PLV,
                        subjects=np.array(SUBJ), bands=np.array(BAND_NAMES),
                        chan_names=np.array(CHAN_NAMES))
    log.info(f'Salvato cache: {CACHE}')

LAB = np.array([CLUSTER_OF[s] for s in SUBJ])
M0, M1 = LAB == 0, LAB == 1
log.info(f'In analisi: C0={M0.sum()}  C1={M1.sum()}')

## §4 — Medie per cluster (C0 e C1)

In [ ]:
# Media entro cluster sui soggetti
POW_C0, POW_C1 = POW[M0].mean(0), POW[M1].mean(0)        # (NB, 61)
PCC_C0, PCC_C1 = PCC[M0].mean(0), PCC[M1].mean(0)        # (NB, 61, 61)
PLV_C0, PLV_C1 = PLV[M0].mean(0), PLV[M1].mean(0)
POW_DIFF = POW_C1 - POW_C0                               # C1 - C0

def cohens_d(a, b, axis=0):
    """d = (mean_b - mean_a) / pooled_std, lungo axis (soggetti)."""
    ma, mb = a.mean(axis), b.mean(axis)
    va, vb = a.var(axis, ddof=1), b.var(axis, ddof=1)
    na, nb = a.shape[axis], b.shape[axis]
    sp = np.sqrt(((na - 1) * va + (nb - 1) * vb) / (na + nb - 2) + 1e-12)
    return (mb - ma) / sp

# Cohen's d per banda×canale sulla band power (C1 vs C0)
D_POW = np.stack([cohens_d(POW[M0][:, k, :], POW[M1][:, k, :]) for k in range(NB)])  # (NB,61)
log.info('Cohen d band power calcolato: ' + str(D_POW.shape))
for k, band in enumerate(BAND_NAMES):
    top = np.argsort(-np.abs(D_POW[k]))[:3]
    log.info(f'  {band:6s}: ' + ', '.join(f'{CHAN_NAMES[i]} d={D_POW[k,i]:+.2f}' for i in top))

## §5 — Topomap della band power: C0, C1, differenza (C1−C0)

Griglia 5 bande × 3 colonne. La terza colonna (differenza) è la firma spaziale che
distingue i due fenotipi per banda.

In [ ]:
info_topo = make_topo_info()
fig, axes = plt.subplots(NB, 3, figsize=(9, 3 * NB))
fig.suptitle('Band power relativa per fenotipo — media soggetto\n'
             f'C0 Fronto-motor (n={M0.sum()})   |   C1 Fronto-occipital (n={M1.sum()})',
             fontsize=13, fontweight='bold')
col_titles = ['C0', 'C1', 'C1 \u2212 C0']
for k, band in enumerate(BAND_NAMES):
    vmax = max(POW_C0[k].max(), POW_C1[k].max())
    dmax = np.abs(POW_DIFF[k]).max()
    for j, (vals, cmap, lim) in enumerate([
            (POW_C0[k], 'viridis', (0, vmax)),
            (POW_C1[k], 'viridis', (0, vmax)),
            (POW_DIFF[k], 'RdBu_r', (-dmax, dmax))]):
        ax = axes[k, j]
        im, _ = mne.viz.plot_topomap(vals, info_topo, axes=ax, show=False,
                                     cmap=cmap, vlim=lim, contours=4)
        if k == 0:
            ax.set_title(col_titles[j], fontsize=11, fontweight='bold')
        if j == 0:
            ax.text(-0.35, 0.5, band, transform=ax.transAxes, fontsize=12,
                    fontweight='bold', va='center', ha='right')
        plt.colorbar(im, ax=ax, fraction=0.045, pad=0.04)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg40_bandpower_topomaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvato: eeg40_bandpower_topomaps.png')

## §6 — Connettività media per banda: |PCC| (ampiezza) e PLV (fase)

Per ciascuna metrica mostriamo la differenza C1−C0 per banda come matrice 61×61, e i top
edge che separano i fenotipi. Attesa: |PCC| separa C0, PLV separa C1.

In [ ]:
def top_edges(diff_mat, n=8):
    iu = np.triu_indices(N_CHAN, k=1)
    vals = diff_mat[iu]
    order = np.argsort(-np.abs(vals))[:n]
    return [(CHAN_NAMES[iu[0][o]], CHAN_NAMES[iu[1][o]], float(vals[o])) for o in order]

for metric_name, C0m, C1m in [('|PCC|', PCC_C0, PCC_C1), ('PLV', PLV_C0, PLV_C1)]:
    fig, axes = plt.subplots(1, NB, figsize=(4 * NB, 4.2))
    fig.suptitle(f'Connettività {metric_name}: differenza C1 \u2212 C0 per banda',
                 fontsize=13, fontweight='bold')
    for k, band in enumerate(BAND_NAMES):
        diff = C1m[k] - C0m[k]
        dmax = np.abs(diff).max() + 1e-9
        ax = axes[k]
        im = ax.imshow(diff, cmap='RdBu_r', vmin=-dmax, vmax=dmax)
        ax.set_title(f'{band}', fontsize=11, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        edges = top_edges(diff, n=5)
        log.info(f'{metric_name} {band:6s} top edge: ' +
                 ', '.join(f'{a}-{b}({v:+.3f})' for a, b, v in edges))
    plt.tight_layout()
    stem = 'pcc' if metric_name == '|PCC|' else 'plv'
    plt.savefig(FIG_DIR / f'eeg40_conn_{stem}_diff.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Salvato: eeg40_conn_{stem}_diff.png')

## §7 — Effect size: quale banda/metrica separa di più i fenotipi?

Per ogni banda riassumiamo la magnitudine della separazione C0 vs C1 (Cohen's d mediano
sui canali per la power; |d| medio sugli edge per la connettività). *Effect size, non p-value.*

In [ ]:
def edge_d(arr, k):
    """Cohen's d per edge (triu) fra C0 e C1 per banda k, poi |d| medio."""
    iu = np.triu_indices(N_CHAN, k=1)
    a = arr[M0][:, k][:, iu[0], iu[1]]   # (n0, n_edge)
    b = arr[M1][:, k][:, iu[0], iu[1]]   # (n1, n_edge)
    d = cohens_d(a, b, axis=0)
    return np.abs(d).mean(), np.abs(d).max()

rows = []
for k, band in enumerate(BAND_NAMES):
    pow_d = np.abs(D_POW[k])
    pcc_m, pcc_mx = edge_d(PCC, k)
    plv_m, plv_mx = edge_d(PLV, k)
    rows.append((band, np.median(pow_d), pow_d.max(), pcc_m, pcc_mx, plv_m, plv_mx))

summary = pd.DataFrame(rows, columns=['band', 'pow_d_med', 'pow_d_max',
                                      'pcc_d_mean', 'pcc_d_max', 'plv_d_mean', 'plv_d_max'])
summary.to_csv(FIG_DIR / 'eeg40_effectsize_summary.csv', index=False)
print(summary.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
xp = np.arange(NB); w = 0.27
ax.bar(xp - w, summary['pow_d_max'],  w, label='band power (|d| max ch)',  color='#59a14f')
ax.bar(xp,     summary['pcc_d_max'],  w, label='|PCC| (|d| max edge)',     color='#4e79a7')
ax.bar(xp + w, summary['plv_d_max'],  w, label='PLV (|d| max edge)',       color='#f28e2b')
ax.axhline(0.2, ls='--', c='gray', lw=0.8); ax.axhline(0.5, ls='--', c='gray', lw=0.8)
ax.text(NB - 0.5, 0.21, "'small'", fontsize=8, color='gray')
ax.text(NB - 0.5, 0.51, "'medium'", fontsize=8, color='gray')
ax.set_xticks(xp); ax.set_xticklabels(BAND_NAMES)
ax.set_ylabel("Cohen's d (C0 vs C1)"); ax.legend()
ax.set_title('Separazione dei fenotipi per banda e metrica — effect size', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg40_effectsize_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvato: eeg40_effectsize_bars.png')

## §8 — Figura riassuntiva per tesi §6.5 + slide *mechanisms*

Una figura pulita: per ciascun fenotipo, la topomap della banda più caratterizzante
(scelta automaticamente dal max |d| in §7) affiancata alla sua firma di connettività.

In [ ]:
# Banda con la maggiore separazione in power e in PLV
band_pow = BAND_NAMES[int(np.argmax(summary['pow_d_max']))]
band_plv = BAND_NAMES[int(np.argmax(summary['plv_d_max']))]
kpow = BAND_NAMES.index(band_pow); kplv = BAND_NAMES.index(band_plv)
log.info(f'Banda più discriminante — power: {band_pow} | PLV: {band_plv}')

info_topo = make_topo_info()
fig, axes = plt.subplots(1, 3, figsize=(13, 4.3))
fig.suptitle('Neural phenotypes of imagined speech — subject-level signatures',
             fontsize=13, fontweight='bold')

# (a) differenza band power nella banda più discriminante
dmax = np.abs(POW_DIFF[kpow]).max()
im, _ = mne.viz.plot_topomap(POW_DIFF[kpow], info_topo, axes=axes[0], show=False,
                             cmap='RdBu_r', vlim=(-dmax, dmax), contours=4)
axes[0].set_title(f'Band power {band_pow}: C1 \u2212 C0', fontsize=11)
plt.colorbar(im, ax=axes[0], fraction=0.045, pad=0.04)

# (b) |PCC| diff nella banda power
dpcc = PCC_C1[kpow] - PCC_C0[kpow]; m = np.abs(dpcc).max()
im2 = axes[1].imshow(dpcc, cmap='RdBu_r', vmin=-m, vmax=m)
axes[1].set_title(f'|PCC| {band_pow}: C1 \u2212 C0 (amplitude)', fontsize=11)
axes[1].set_xticks([]); axes[1].set_yticks([])
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

# (c) PLV diff nella banda fase
dplv = PLV_C1[kplv] - PLV_C0[kplv]; m = np.abs(dplv).max()
im3 = axes[2].imshow(dplv, cmap='RdBu_r', vmin=-m, vmax=m)
axes[2].set_title(f'PLV {band_plv}: C1 \u2212 C0 (phase)', fontsize=11)
axes[2].set_xticks([]); axes[2].set_yticks([])
plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg40_phenotype_summary.png', dpi=160, bbox_inches='tight')
plt.show()
print('Salvato: eeg40_phenotype_summary.png')

## §9 — Export risultati

Salva le medie per cluster e la tabella effect-size per la tesi (§6.5) e per il pptx storico.

In [ ]:
OUT = CKPT_DIR / 'cluster_band_profiles.npz'
np.savez_compressed(OUT,
                    pow_c0=POW_C0, pow_c1=POW_C1, pow_diff=POW_DIFF,
                    pcc_c0=PCC_C0, pcc_c1=PCC_C1,
                    plv_c0=PLV_C0, plv_c1=PLV_C1,
                    d_pow=D_POW,
                    bands=np.array(BAND_NAMES), chan_names=np.array(CHAN_NAMES),
                    n_c0=int(M0.sum()), n_c1=int(M1.sum()))
print(f'Salvato: {OUT}')
print('\nFigure prodotte in figures/:')
for f in ['eeg40_bandpower_topomaps', 'eeg40_conn_pcc_diff', 'eeg40_conn_plv_diff',
          'eeg40_effectsize_bars', 'eeg40_phenotype_summary']:
    print(f'  - {f}.png')
print('  - eeg40_effectsize_summary.csv')